In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
from pathlib import Path
import sys
%matplotlib inline

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
# read in all the words
DATA_PATH = ROOT / "data" / "names.txt"
words = open(DATA_PATH, 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
# single char - 27 possibility of context - 27,27
# two char - [27,27] - 27,27,27 matrix
len(words)

32033

In [21]:
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [101]:
# creaate dataset

block_size = 3 # context lenght
X, Y = [], [] # x input, y is label

for w in words[:5]:
    print(w)
    context = [0] * block_size 

    for ch in w + ".":
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print("".join(itos[i] for i in context), "--->", itos[ix])
        print(context, ix)
        context = context[1:] + [ix]  # crop and append, rolling window

        
X = torch.tensor(X)
Y = torch.tensor(Y)
X,Y

emma
... ---> e
[0, 0, 0] 5
..e ---> m
[0, 0, 5] 13
.em ---> m
[0, 5, 13] 13
emm ---> a
[5, 13, 13] 1
mma ---> .
[13, 13, 1] 0
olivia
... ---> o
[0, 0, 0] 15
..o ---> l
[0, 0, 15] 12
.ol ---> i
[0, 15, 12] 9
oli ---> v
[15, 12, 9] 22
liv ---> i
[12, 9, 22] 9
ivi ---> a
[9, 22, 9] 1
via ---> .
[22, 9, 1] 0
ava
... ---> a
[0, 0, 0] 1
..a ---> v
[0, 0, 1] 22
.av ---> a
[0, 1, 22] 1
ava ---> .
[1, 22, 1] 0
isabella
... ---> i
[0, 0, 0] 9
..i ---> s
[0, 0, 9] 19
.is ---> a
[0, 9, 19] 1
isa ---> b
[9, 19, 1] 2
sab ---> e
[19, 1, 2] 5
abe ---> l
[1, 2, 5] 12
bel ---> l
[2, 5, 12] 12
ell ---> a
[5, 12, 12] 1
lla ---> .
[12, 12, 1] 0
sophia
... ---> s
[0, 0, 0] 19
..s ---> o
[0, 0, 19] 15
.so ---> p
[0, 19, 15] 16
sop ---> h
[19, 15, 16] 8
oph ---> i
[15, 16, 8] 9
phi ---> a
[16, 8, 9] 1
hia ---> .
[8, 9, 1] 0


(tensor([[ 0,  0,  0],
         [ 0,  0,  5],
         [ 0,  5, 13],
         [ 5, 13, 13],
         [13, 13,  1],
         [ 0,  0,  0],
         [ 0,  0, 15],
         [ 0, 15, 12],
         [15, 12,  9],
         [12,  9, 22],
         [ 9, 22,  9],
         [22,  9,  1],
         [ 0,  0,  0],
         [ 0,  0,  1],
         [ 0,  1, 22],
         [ 1, 22,  1],
         [ 0,  0,  0],
         [ 0,  0,  9],
         [ 0,  9, 19],
         [ 9, 19,  1],
         [19,  1,  2],
         [ 1,  2,  5],
         [ 2,  5, 12],
         [ 5, 12, 12],
         [12, 12,  1],
         [ 0,  0,  0],
         [ 0,  0, 19],
         [ 0, 19, 15],
         [19, 15, 16],
         [15, 16,  8],
         [16,  8,  9],
         [ 8,  9,  1]]),
 tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
          1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0]))

In [102]:
# embedding table C

# 17K words in 30 dimension
# 27char -> 2D space
# 27 row, 2 columns

C = torch.randn((27,2))
emb = C[X]
emb, emb.shape

(tensor([[[ 0.2537, -0.6169],
          [ 0.2537, -0.6169],
          [ 0.2537, -0.6169]],
 
         [[ 0.2537, -0.6169],
          [ 0.2537, -0.6169],
          [-0.7531,  1.2424]],
 
         [[ 0.2537, -0.6169],
          [-0.7531,  1.2424],
          [-0.6700, -1.7309]],
 
         [[-0.7531,  1.2424],
          [-0.6700, -1.7309],
          [-0.6700, -1.7309]],
 
         [[-0.6700, -1.7309],
          [-0.6700, -1.7309],
          [-1.4972, -1.5287]],
 
         [[ 0.2537, -0.6169],
          [ 0.2537, -0.6169],
          [ 0.2537, -0.6169]],
 
         [[ 0.2537, -0.6169],
          [ 0.2537, -0.6169],
          [ 0.8638,  1.0579]],
 
         [[ 0.2537, -0.6169],
          [ 0.8638,  1.0579],
          [-0.0155, -0.1038]],
 
         [[ 0.8638,  1.0579],
          [-0.0155, -0.1038],
          [-1.0663, -0.4513]],
 
         [[-0.0155, -0.1038],
          [-1.0663, -0.4513],
          [-0.6576, -0.5781]],
 
         [[-1.0663, -0.4513],
          [-0.6576, -0.5781],
          

In [103]:
X[13,2]

tensor(1)

In [104]:
C[X][13,2]

tensor([-1.4972, -1.5287])

In [105]:
C[1]

tensor([-1.4972, -1.5287])

In [106]:
# hidden layer
W1 = torch.randn(6,100)
b1 = torch.randn(100)

# • W1 is [6, 100] because the first linear layer maps:

#   6 input features -> 100 hidden features

#   Here is where the 6 comes from.

#   You have:

#   - context length = 3
#   - embedding size = 2

#   So each example has 3 characters, and each character becomes a 2D embedding.


In [107]:
# what we want to do is emb @ W1 + b1
# but W1 is 6,100 and emb is 32,3,2 - so we need to concatenate so we can mul
# how do we transform is?
# torch.cat(torch.unbind(emb,1),1) or emb.view(32,6) 
# same resullt but view is more effecient - https://blog.ezyang.com/2019/05/pytorch-internals/
# as cat createes new memeory, new storage etc
emb.view(32,6)


tensor([[ 0.2537, -0.6169,  0.2537, -0.6169,  0.2537, -0.6169],
        [ 0.2537, -0.6169,  0.2537, -0.6169, -0.7531,  1.2424],
        [ 0.2537, -0.6169, -0.7531,  1.2424, -0.6700, -1.7309],
        [-0.7531,  1.2424, -0.6700, -1.7309, -0.6700, -1.7309],
        [-0.6700, -1.7309, -0.6700, -1.7309, -1.4972, -1.5287],
        [ 0.2537, -0.6169,  0.2537, -0.6169,  0.2537, -0.6169],
        [ 0.2537, -0.6169,  0.2537, -0.6169,  0.8638,  1.0579],
        [ 0.2537, -0.6169,  0.8638,  1.0579, -0.0155, -0.1038],
        [ 0.8638,  1.0579, -0.0155, -0.1038, -1.0663, -0.4513],
        [-0.0155, -0.1038, -1.0663, -0.4513, -0.6576, -0.5781],
        [-1.0663, -0.4513, -0.6576, -0.5781, -1.0663, -0.4513],
        [-0.6576, -0.5781, -1.0663, -0.4513, -1.4972, -1.5287],
        [ 0.2537, -0.6169,  0.2537, -0.6169,  0.2537, -0.6169],
        [ 0.2537, -0.6169,  0.2537, -0.6169, -1.4972, -1.5287],
        [ 0.2537, -0.6169, -1.4972, -1.5287, -0.6576, -0.5781],
        [-1.4972, -1.5287, -0.6576, -0.5

In [108]:
# now we can do :
# emb.view(32,6) @ W1 + b1
# instead of 32, we could do emb.shape[0] for number of example or -1 
#(emb.view(-1,6) @ W1 + b1)
# we want tanh of it so
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
h, h.shape

(tensor([[ 0.7918,  0.8093,  0.9248,  ..., -0.4811, -0.9693, -0.9915],
         [ 0.9429, -0.6624, -0.9567,  ...,  0.8722,  0.9746,  0.1677],
         [ 0.9948,  1.0000,  0.9999,  ...,  0.4516, -0.9997, -0.9981],
         ...,
         [ 0.9988,  0.9529,  0.9823,  ..., -0.9840, -0.9970, -0.4261],
         [ 0.9993,  0.9988,  0.3920,  ..., -0.7602, -0.9966, -0.9999],
         [ 0.0049,  0.9999, -0.8879,  ...,  0.9874, -0.9999, -1.0000]]),
 torch.Size([32, 100]))

In [109]:
# also broadcasting is happening
(emb.view(-1,6) @ W1).shape, b1.shape

(torch.Size([32, 100]), torch.Size([100]))

In [110]:
# 32, 100
#   , 100 gets broadcasted to -> 1,100, aling on right
# so it adds bias vector to  row

In [111]:
W2 = torch.randn(100,27)
b2 = torch.randn(27)
W2, b2

(tensor([[ 1.0208, -1.9581, -0.5844,  ..., -1.2963, -0.9072, -0.0958],
         [ 0.1982, -0.9168, -0.5479,  ..., -0.1982,  0.2190, -0.7127],
         [-0.6845,  0.4896, -0.1181,  ..., -0.1167,  0.0064,  1.0510],
         ...,
         [ 0.2486, -1.1958,  0.4690,  ..., -0.0557, -0.1810, -0.1273],
         [ 0.0218, -0.4185,  0.1715,  ..., -0.4252,  0.0356,  0.1825],
         [-3.3118, -1.1494,  1.3102,  ..., -0.2738,  0.5328,  0.5720]]),
 tensor([ 0.4678, -0.8975, -1.0859, -0.2532,  0.1918,  0.0056,  0.9118, -0.2158,
          1.2457, -0.4829,  0.9618, -3.5015,  0.3348,  1.2384, -0.4007, -0.6049,
         -0.8666, -0.3120,  1.3745,  0.2148, -0.7519,  1.7007, -0.1861,  1.9494,
          1.1273, -0.1419, -1.4471]))

In [112]:
logits = h @ W2 + b2
logits.shape

torch.Size([32, 27])

In [113]:
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)
probs.shape, probs[0].sum()

(torch.Size([32, 27]), tensor(1.))

In [114]:
torch.arange(32),range(len(Y))

(tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]),
 range(0, 32))

In [115]:
# So, what we would like to do now is index into the rows of prob and for 
# each row to pluck out the probability given to the correct character:
probs[torch.arange(32), Y]

tensor([6.0030e-11, 7.3689e-03, 8.8270e-05, 9.9104e-09, 9.5191e-02, 6.5953e-11,
        7.8140e-09, 3.0628e-08, 1.0217e-16, 2.3769e-05, 4.7864e-08, 2.9643e-02,
        9.9528e-01, 8.1269e-14, 9.0863e-01, 1.5937e-02, 2.4777e-07, 5.2577e-06,
        4.0965e-04, 7.9980e-04, 1.6731e-06, 2.7684e-08, 3.7136e-03, 1.2657e-10,
        1.6790e-01, 1.3659e-08, 3.8107e-09, 3.4095e-09, 3.1120e-04, 9.0593e-15,
        7.2737e-03, 2.4294e-03])

In [116]:
#This gives the current probabilities for these specific correct, target characters that come next after 
# each character sequence,given the current nn configuration (weights and biases). 
# Currently these probabilities are pretty bad and most characters are pretty unlikely to occur next. 
# ideall all thesee prob are 1, since it is from the training

In [117]:
loss = -probs[torch.arange(32), Y].log().mean()
loss

tensor(13.4910)

In [118]:
parameters = [C, W1, b1, W2, b2]

In [119]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [120]:
for p in parameters:
  p.requires_grad = True

In [131]:
for _ in range(100): 
    # forward pass
    emb = C[X]
    h = torch.tanh(emb.view(-1,6) @ W1 + b1)  # [32, 100]
    logits = h @ W2 + b2  # [32, 27]
    # counts = logits.exp()
    # prob = counts / counts.sum(1, keepdims=True)
    # loss = -probs[torch.arange(32), Y].log().mean()
    loss = F.cross_entropy(logits,Y) # we always use that, no need to create intermediate tensors, backward pass is more effecient, handles very positive - run out of range - offset/sub all with max value
    print(loss)
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    for p in parameters:
        p.data += -0.1 * p.grad


tensor(0.2579, grad_fn=<NllLossBackward0>)
tensor(0.2579, grad_fn=<NllLossBackward0>)
tensor(0.2579, grad_fn=<NllLossBackward0>)
tensor(0.2579, grad_fn=<NllLossBackward0>)
tensor(0.2579, grad_fn=<NllLossBackward0>)
tensor(0.2579, grad_fn=<NllLossBackward0>)
tensor(0.2579, grad_fn=<NllLossBackward0>)
tensor(0.2579, grad_fn=<NllLossBackward0>)
tensor(0.2579, grad_fn=<NllLossBackward0>)
tensor(0.2579, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.2578, grad_fn=<NllLossBackward0>)
tensor(0.25

In [127]:
logits.max(1) # loss is not exactly 0, we are not able to overfit

torch.return_types.max(
values=tensor([ 9.0702, 11.9129, 12.0175, 19.3679, 16.0234,  9.0702, 15.3041, 16.6228,
        16.4572, 11.6610, 16.8509, 14.6982,  9.0702, 11.8484, 18.5098, 17.8150,
         9.0702, 13.8731, 15.2275, 12.6977, 18.1343, 19.6995, 13.3820, 11.0570,
        10.7118,  9.0702, 12.7961, 15.7459, 20.8289, 13.8611, 17.5929, 18.2389],
       grad_fn=<MaxBackward0>),
indices=tensor([19, 13, 13,  1,  0, 19, 12,  9, 22,  9,  1,  0, 19, 22,  1,  0, 19, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0]))

In [128]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [130]:
# minibatch explanation
# Same, the loss for all input samples also keeps decreasing. But, you’ll notice that training takes longer now. This is happening because we are doing a lot of work, forward and backward passing on 228146 examples. That’s way too much work! In practice, what people usually do in such cases is they train on minibatches of the whole dataset. So, what we want to do, is we want to randomly select some portion of the dataset, and that’s a minibatch! And then, only forward, backward and update on that minibatch, likewise iterate and train on those minibatches. A simple way to implement minibatching is to set a batch size, e.g.:

# batchsize = 32

# and then to randomly select batchsize number of indeces referencing the subset of input data to be used for minibatch training. To get the indeces you can do something like this:

# batchix = torch.randint(0, x_all.shape[0], (batchsize,))
# print(batchix)

# tensor([ 74679, 122216,  57092, 133769, 226181,  38045, 126099,  23446, 218663,
#          17662, 225764, 199486, 185049,  64041, 217855, 198821, 192633,  84825,
#          44722,  46171, 182390,  99196, 102624,    409, 168159, 182770, 142590,
#         173184,  86521,   1596, 158516, 206175])
# Then, to actually get a minibatch per epoch, just create a new, random set of indeces and index the samples and targets from the dataset before each forward pass. Like th